# Capability 25: Graceful handling of unsupported or unavailable requests

8/8 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 25`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap25_graceful_handling.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: unsupported entity (real competitor) -> clear message, not a crash

**✅ PASS**

In [ ]:
_ = ask("What was Carlsberg's revenue last year?", '01')

01
USER: What was Carlsberg's revenue last year?

[intent=data_query | sub_agents=['structured']]

AGENT: I don’t have that figure in the retrieved evidence — the KPI query returned no matching rows for Carlsberg's revenue, so I can’t confirm or estimate last year’s revenue from available data.

If you can share the source or reporting period, I can help validate it. Alternatively, I can pull AB InBev revenue or other available brewery KPIs from the provided datasets.

## 02: unsupported granularity (brand-level financials) -> clear message, not a fabricated number

**✅ PASS**

In [ ]:
_ = ask("What was Corona's exact global revenue figure in 2025?", '02')

02
USER: What was Corona's exact global revenue figure in 2025?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-004', 'DOC-014', 'DOC-008', 'DOC-011']]

AGENT: Based on the retrieved evidence, I’m unable to provide an exact 2025 global revenue figure for Corona in USD millions.

What the evidence does show:

- AB InBev’s “megabrands” — which include Corona, Budweiser, Stella Artois, and Michelob Ultra — grew revenue **4.1% in FY2025** [DOC-012].
- The same document notes that the Corona figures cited **exclude Mexico**, where Constellation Brands holds a permanent license to the Corona/Modelo brand [DOC-012].
- No document in the provided evidence contains a standalone Corona global revenue number for 2025 [DOC-012][DOC-008][DOC-004][DOC-011].

So the exact figure you asked for is **not supported by the evidence provided**. If you’re looking for that specific number, it would likely need to come from a brand-level revenue disclosure or a more granular ea

## 03: a nonsensical/garbled question -> handled gracefully, not a crash

**✅ PASS**

In [ ]:
_ = ask('asdkfj revenue zzz???', '03')

03
USER: asdkfj revenue zzz???

[intent=clarification_needed | sub_agents=[]]

AGENT: Which zone and period would you like revenue for?

## 04: an empty-ish/minimal question -> handled gracefully

**✅ PASS**

In [ ]:
_ = ask('?', '04')

04
USER: ?

[intent=clarification_needed | sub_agents=[]]

AGENT: What would you like to know about AB InBev's financial performance?

## 05: a request for a data granularity that flatly doesn't exist (channel-level)

**✅ PASS**

In [ ]:
_ = ask("What was North America's e-commerce channel revenue specifically?", '05')

05
USER: What was North America's e-commerce channel revenue specifically?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-004', 'DOC-012', 'DOC-003', 'DOC-011', 'DOC-001']]
[assumptions/limitations surfaced:]
  - 'e-commerce channel' isn't part of Anheuser-Busch InBev (AB InBev)'s tracked entities (brand/zone/country), or is a different company entirely (e.g. a competitor), so no internal structured data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting.

AGENT: Based on the retrieved AB InBev internal reporting documents, I can't give you a specific e-commerce channel revenue figure for North America. E-commerce is not a tracked or reported segment in the internal KPI data available — the tracked dimensions are zone, brand, and country, with revenue reported at the zone level (e.g., North America total revenue), not broken out by sales channel.

For reference, North America's **total revenue** figures in 

## 06: a request mixing a supported and unsupported entity in one turn

**✅ PASS**

In [ ]:
_ = ask("Compare AB InBev's North America revenue to Molson Coors'", '06')

06
USER: Compare AB InBev's North America revenue to Molson Coors'

[intent=comparison | sub_agents=['structured', 'web']]
[assumptions/limitations surfaced:]
  - 'Molson Coors' isn't part of Anheuser-Busch InBev (AB InBev)'s tracked entities (brand/zone/country), or is a different company entirely (e.g. a competitor), so no internal structured data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting.

AGENT: ## Short answer

I can't produce a like-for-like revenue comparison here — the evidence needed for it isn't available.

**What the evidence shows:**

| Source | Result |
|---|---|
| Internal structured KPIs (`fact_kpi`, zone = North America, kpi = revenue_usd_m) | **No matching rows returned** — the query came back empty |
| Web search | **No results** provided |
| Internal documents | **None** supplied |

**Why the comparison can't be made:**

1. **No AB InBev North America figure is in hand.** The structured KPI query for 

## 07: out-of-scope request handled gracefully rather than forced into a data answer

**✅ PASS**

In [ ]:
_ = ask("What's the capital of France?", '07')

07
USER: What's the capital of France?

[intent=out_of_scope | sub_agents=[]]

AGENT: That's outside what I can help with -- I'm scoped to Anheuser-Busch InBev (AB InBev)'s business data and related market/company context. Anheuser-Busch InBev (AB InBev)'s real, publicly disclosed financial performance -- revenue, volume, normalized EBITDA, EBITDA margin, organic revenue growth and net profit -- by reporting zone (North America, Middle Americas, South America, EMEA, Asia Pacific) and by quarter or year, plus qualitative country- and brand-level commentary, company news, and competitive context drawn from AB InBev's own results releases and filings.

## 08: a request for a period outside the known data range

**✅ PASS**

In [ ]:
_ = ask("What was North America's revenue in Q1 2030?", '08')

08
USER: What was North America's revenue in Q1 2030?

[intent=data_query | sub_agents=['structured']]

AGENT: I couldn't find a matching record for **North America revenue in Q1 2030** in the available structured KPI data — the query returned no rows for that zone/period combination.

This may mean the data point hasn't been loaded yet, or that the period/zone combination isn't currently tracked in the source.

If you'd like, I can check:

- **North America revenue for Q1 2029** or another available quarter
- Revenue for **another zone in Q1 2030**